# Short-Video Analysis and Temporal Evaluation

**LLM Agents and Video Analysis · guided lesson**  
**Plan for:** 60–90 minutes  
**Code runtime:** live RCD requests; Clemson network or CUVPN required

## Learning outcomes

- Validate media before transmission and discover video-capable models.
- Construct a structured short-video request.
- Separate syntax, schema, temporal matching, and semantic support.
- Identify privacy, sampling, timestamp, and unsupported-inference risks.

| Segment | Suggested minutes |
|---|---:|
| Motivation and mental model | 15–20 |
| Guided implementation | 35–45 |
| Failure analysis and exercise | 15–20 |
| Summary and homework | 5 |
| **Total** | **60–90** |

## Driving question

> **How can we distinguish a well-formed video response from one that is temporally and visually supported?**

You need Python and basic API familiarity. The RCD endpoint is the classroom
service, and Clemson network or CUVPN access is required. Each concept below is
followed immediately by the code and evidence used to test it.

## How this lesson builds, step by step

1. **Validate an approved local clip and fixed model.**
2. **Construct a bounded multimodal prompt.**
3. **Send the complete video to the local Omni model.**
4. **Compare thinking and non-thinking output budgets.**
5. **Parse and validate structured JSON.**

If a result differs from your prediction, stop at that boundary before continuing.

## Work the smallest useful example

A reference event spans seconds 4–10 and a predicted event spans 6–12. Their temporal intersection is 4 seconds and their union is 8 seconds, so temporal IoU is `4/8=.5`. This supports a possible match but says nothing about whether the descriptions refer to the same visible action. If a model emits valid JSON with chronological ranges, only syntactic and structural checks have passed.


In [ ]:
from pathlib import Path
import os
import sys

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "course_helpers.py").is_file():
        COURSE_ROOT = candidate.resolve()
        break
else:
    raise RuntimeError("Open this notebook from the course directory or its notebooks/ folder.")

os.chdir(COURSE_ROOT)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))
print("Course root:", COURSE_ROOT)


In [ ]:
import base64
import json
from pathlib import Path

VIDEO_MIME_TYPES = {
    ".mp4": "video/mp4", ".webm": "video/webm",
    ".mov": "video/quicktime",
}

def validate_video_path(video_path, max_bytes=15 * 1024 * 1024):
    path = Path(video_path).expanduser().resolve()
    if not path.is_file():
        raise ValueError(f"Video not found: {path}")
    mime = VIDEO_MIME_TYPES.get(path.suffix.lower())
    if mime is None:
        raise ValueError(f"Unsupported video suffix: {path.suffix}")
    if path.stat().st_size == 0 or path.stat().st_size > max_bytes:
        raise ValueError("Video is empty or exceeds the classroom limit.")
    return path, mime

def video_data_url(video_path):
    path, mime = validate_video_path(video_path)
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"data:{mime};base64,{encoded}"


These first versions make the important operations visible: validate before
reading bytes, map an allowlisted suffix to a MIME type, and explicitly
encode the approved file. The API key and encoded video must never be
printed.

The maintained module versions add clearer exceptions, response parsing,
and schema checks. After understanding the code above, we import them from
`course_helpers.py` so later cells focus on the experiment rather than
repeat infrastructure.


In [ ]:
from course_helpers import (
    VIDEO_ANALYSIS_PROMPT, VIDEO_MODEL, VideoValidationError,
    analyze_video, compare_events, create_client, require_models,
    validate_video_analysis, validate_video_path,
)

client = create_client()
require_models([VIDEO_MODEL], client=client)
print("Direct-video model:", VIDEO_MODEL)


<details><summary><strong>Code walkthrough: selecting a video model</strong></summary>

The direct-video path is fixed to `qwen3-omni-30b-a3b` on the local RCD
endpoint. Setup checks the exact ID; the lesson never infers video support
from a vague model name or silently substitutes a text-only model.

</details>


### A video answer passes through several review layers

Local media checks happen before transmission; syntax, timing, and semantic support are checked after generation.

**Check your understanding:** Which check can reject malformed JSON, and which check can catch a fluent but visually unsupported claim?

## 1. What the model actually receives

Video models typically sample frames rather than inspect every moment continuously. Small actions may be missed, and timestamp precision can exceed the evidence. Audio understanding is not assumed in this core exercise.

<img src="../assets/figures/video-analysis-pipeline.svg" alt="Video-analysis pipeline from local validation and encoding through a video model, JSON validation, and human review" width="960">

A direct video request packages media for the model, but model internals may decode, resize, and sample it. Consequently:

- short or visually subtle events may fall between sampled frames;
- cuts, overlays, and camera motion can be confused with scene events;
- exact timestamps may be inferred from coarse evidence;
- visible actions do not necessarily reveal intent, identity, or causality;
- audio capability must be checked separately rather than assumed from `video` support.

**Predict before running:** Which event types are most likely to be missed by sparse frame sampling?


## Learn and test: Validate an approved local clip and fixed model.

A video endpoint may sample frames, resize them, and omit audio. Brief events can disappear between samples, while camera motion can resemble object motion. Output timestamps may therefore look more precise than the available evidence.


### Follow data across the application boundary

The diagram separates local notebook work, the request sent to the service, and application-owned validation.

**Check your understanding:** Which information crosses the network, and which files remain local unless the code explicitly sends them?

<img src="../assets/figures/course-data-flow.svg" alt="Course-original figure: The diagram separates local notebook work, the request sent to the service, and application-owned validation." width="960">


### Try it: Validate an approved local clip and fixed model.

**What this cell shows:** Validate an approved local clip and fixed model.

**What goes in and comes out:** A local NASA clip, byte limit, MIME allowlist, and fixed Omni model enter; a validated path and media type come out.

**Before you run the cell:** Check existence, suffix, MIME type, nonzero size, and classroom size limit before reading bytes.


In [ ]:
video_path = Path("assets/nasa_three_pacific_hurricanes.mp4")
path, mime = validate_video_path(video_path)
print(f"Ready: {path.name} ({path.stat().st_size / 1024 / 1024:.2f} MB, {mime})")

**What you should see:** The approved clip is ready for encoding and the exact model ID was checked in setup.

**If your result looks different:** A readable file is not automatically permitted data; classification and licensing remain separate checks.


<details><summary><strong>Code walkthrough: validating before encoding</strong></summary>

`Path` resolves the course-relative video location. `validate_video_path()` checks existence, an allowlisted suffix/MIME mapping, nonzero size, and the classroom byte limit before reading the whole file. These checks avoid expensive base64 work and produce specific feedback close to the source of the error.

Validation does not inspect copyright, consent, or data classification; those are instructor and researcher responsibilities.

</details>


## 2. Direct multimodal request

The helper base64-encodes the local clip as a `data:video/...` URL and sends a Chat Completions message with two content items:

```python
[
    {"type": "video_url", "video_url": {"url": "data:video/mp4;base64,<omitted>"}},
    {"type": "text", "text": VIDEO_ANALYSIS_PROMPT},
]
```

The notebook never prints the encoded video or API key.

Base64 increases the transmitted representation beyond the original file size, so the course enforces a conservative pre-encoding limit. File validation protects classroom reliability; it does not determine whether the video is ethically or legally appropriate to submit. Data classification and permission must be checked before the request.

Qwen3-Omni can produce a separate reasoning channel. We will run the same model
twice: once with thinking disabled and once with thinking enabled. The first
request asks for only 1,200 new tokens. The thinking request allows 4,096 because
its budget must hold both internal reasoning and the final JSON answer.

In this OpenAI-compatible endpoint, `max_tokens` means the maximum number of
**new output tokens**. It plays the role often called `max_new_tokens` in local
Hugging Face generation code. If the thinking budget is too small, the model may
use it all before producing `message.content`.


### Structured prompt

Read the prompt below. Notice that it asks for observable evidence, chronological events, approximate timestamps, and uncertainties.


## Learn and test: Construct a bounded multimodal prompt.

Validate locally before transmission: path, extension, MIME type, size, duration when available, and data classification. Use public, course-authored, or explicitly approved media. A base64 data URL copies the complete clip into the request, so size limits and privacy review occur before encoding. Capability discovery prevents accidentally sending video to a text-only model.


### A conversation must fit inside one context window

Instructions, history, the current request, and the answer all compete for a bounded token budget.

**Check your understanding:** If the history grows, what can the application summarize or remove without losing the current constraint?

<img src="../assets/figures/context-budget.svg" alt="Course-original figure: Instructions, history, the current request, and the answer all compete for a bounded token budget." width="960">


### Try it: Construct a bounded multimodal prompt.

**What this cell shows:** Construct a bounded multimodal prompt.

**What goes in and comes out:** Validated media metadata and a focused prompt enter; a size-bounded request comes out.

**Before you run the cell:** Inspect the media reference, requested schema, time range, and maximum output budget.


In [ ]:
print(VIDEO_ANALYSIS_PROMPT)

**What you should see:** The request asks only for observable events and preserves the selected clip identity.

**If your result looks different:** If the request invites intent or identity inference, later schema validation cannot repair that risk.


### Experiment A — non-thinking video analysis

Start with the direct path. It is usually faster and a modest output budget is
enough for this short JSON schema. Before running the cell, predict how many
events the model will identify and which claims should appear under
`uncertainties` rather than as observations.


## Learn and test: Send the complete video to the local Omni model.

Structured output helps downstream handling but does not establish truth. Syntax
checks JSON; schema checks fields and ranges. Temporal evaluation identifies
missing or extra events. Semantic review asks whether visible evidence supports
each description.


### Try it: Send the complete video to the local Omni model.

**What this cell shows:** Send the complete video to the local Omni model.

**What goes in and comes out:** The packaged NASA MP4 and fixed Qwen Omni model enter; one live structured analysis record comes out.

**Before you run the cell:** Confirm the exact model ID, complete-video content item, token budget, and returned JSON fields.


In [ ]:
analysis_nonthinking = analyze_video(
    video_path, VIDEO_ANALYSIS_PROMPT, VIDEO_MODEL,
    client=client,
    max_tokens=1200,
    enable_thinking=False,
)
print(json.dumps(analysis_nonthinking, indent=2))

**What you should see:** The live result is ready for structural and evidence review.

**If your result looks different:** A valid JSON response still does not prove that every event is visually supported.


Check that the result is a JSON object with `summary`, `events`, and
`uncertainties`. Then compare each event with the video itself: the model's
confidence does not make an event visually supported.


### Experiment B — thinking video analysis

Now enable Qwen's thinking mode while keeping the video and prompt unchanged.
We increase `max_tokens` from 1,200 to 4,096. This is not a request for a longer
final answer; it reserves space for reasoning **and** the final JSON. If the
server stops with no final content, increase this value further rather than
trying to parse the reasoning channel as the answer.


## Learn and test: Compare thinking and non-thinking output budgets.

Position-by-position event matching fails when a model merges or splits events. Temporal IoU provides a candidate association, followed by human review of descriptions and evidence. Report matched, missing, extra, shifted, and unsupported events separately. Do not collapse them into one accuracy number without a documented matching policy.


### Try it: Compare thinking and non-thinking output budgets.

**What this cell shows:** Compare thinking and non-thinking output budgets.

**What goes in and comes out:** The same video and Qwen3-Omni model enter twice with thinking disabled and enabled.

**Before you run the cell:** Hold the prompt fixed and compare the 1,200-token direct budget with the 4,096-token thinking budget.


In [ ]:
analysis_thinking = analyze_video(
    video_path, VIDEO_ANALYSIS_PROMPT, VIDEO_MODEL,
    client=client,
    max_tokens=4096,
    enable_thinking=True,
)
print(json.dumps(analysis_thinking, indent=2))

**What you should see:** Both requests should end with a parseable final JSON answer.

**If your result looks different:** A thinking request can exhaust its budget before producing final content.


Compare the two JSON records claim by claim. Thinking is not automatically
more accurate: verify timestamps, unsupported inferences, and omitted events
in both outputs. The later evaluation uses the non-thinking result as its fixed
baseline so that rerunning this optional comparison does not change the lesson.


<details><summary><strong>Code walkthrough: sending video and parsing the answer</strong></summary>

`analyze_video()` performs seven steps:

1. validate the local path and size;
2. base64-encode the clip into a `data:video/...` URL;
3. place a `video_url` item and text prompt in one user message;
4. call `qwen3-omni-30b-a3b` through the local RCD Chat Completions endpoint;
5. explicitly enable or disable thinking instead of relying on the server default;
6. separate the reasoning and final-answer channels, then parse only the final answer as JSON;
7. validate its required fields and chronological time ranges.

If the thinking cell reports reasoning but no final answer, raise `max_tokens`.
If an old cell still raises `AttributeError: 'NoneType' ... strip`, restart the
kernel and rerun the setup cells so Python reloads the updated helper module.

</details>


### Compare predicted and reference events in time

The two timelines reveal shifted boundaries, missing events, and extra events that one aggregate score could hide.

**Check your understanding:** Which pair has enough temporal overlap to inspect semantically, and which event has no matching prediction?

## 3. Validate structure, then evaluate meaning

Schema validation catches malformed JSON, missing fields, invalid ranges, and non-chronological events. It cannot prove that the description is visually grounded.

Use three separate review layers:

1. **Syntactic:** Is the response valid JSON?
2. **Structural:** Are required fields and time ranges valid?
3. **Semantic:** Does visible evidence actually support each claim?

<img src="../assets/figures/video-timeline.svg" alt="Human reference and model event timelines showing shifted boundaries and a missing event" width="960">

Event matching is not inherently positional: a model may merge two reference events or split one event into several. The provided comparison table is intentionally a discussion aid rather than an automated accuracy metric.


## Learn and test: Parse and validate structured JSON.

Thinking and non-thinking requests use the same video and model but have different output-budget needs. Production systems also need retry policy, rate-limit handling, privacy-preserving logs, model/version records, and escalation when media is sensitive or conclusions are consequential.


### Try it: Parse and validate structured JSON.

**What this cell shows:** Parse and validate structured JSON.

**What goes in and comes out:** Raw response text enters; parsed events or a named parsing error come out.

**Before you run the cell:** Check required keys, numeric start/end times, chronological order, and bounds within clip duration.


In [ ]:
validated = validate_video_analysis(analysis_nonthinking)
assert validated["summary"]
assert all(event["end_seconds"] >= event["start_seconds"] for event in validated["events"])
print("Structure valid. Semantic review is still required.")

**What you should see:** A passing result establishes syntax and schema only—not visual truth.

**If your result looks different:** Reject extra prose or out-of-range timestamps rather than silently coercing them.


<details><summary><strong>Code walkthrough: structural assertions</strong></summary>

`validate_video_analysis()` checks the full schema and chronology. The two `assert` statements then make the notebook's teaching expectations visible: a nonempty summary and nonnegative event ranges. Assertions are useful for demonstration, but production input validation should raise deliberate exceptions rather than depend only on assertions, which can be disabled.

</details>


In [ ]:
ground_truth = json.loads(Path("assets/video_ground_truth.json").read_text())
comparison = compare_events(validated["events"], ground_truth["events"], tolerance_seconds=2.0)
for row in comparison:
    print(json.dumps(row, ensure_ascii=False))

<details><summary><strong>Code walkthrough: comparison without false certainty</strong></summary>

The human reference is loaded from a separate JSON file so instructors can change the video without editing notebook code. `compare_events()` pairs events for discussion and checks whether start times fall within a tolerance. Every row still carries `requires_human_review: true` because positional matching and timestamp tolerance cannot establish semantic correctness.

</details>


### Audit exercise

Find at least one of the following:

- a missing event;
- an event not supported by visible evidence;
- an inference presented as observation;
- timestamp precision unsupported by frame sampling;
- an uncertainty the model failed to mention.

Do not convert the comparison table into an automatic claim of correctness.


## 4. Failure handling

Invalid paths, unsupported suffixes, empty/oversized clips, unavailable video models, timeouts, and malformed JSON produce readable course errors.

Distinguish **recoverable operational failures** (temporary timeout or rate limit) from **invalid assumptions** (text-only model, unsupported format, or disallowed data). Retrying can help the former; it should not bypass the latter.


In [ ]:
try:
    validate_video_path("assets/not-a-video.txt")
except VideoValidationError as exc:
    print("Expected validation example:", exc)

## Checkpoint and responsible-use wrap-up

- Exact lesson model IDs must be checked before the first request.
- Short videos reduce cost and failure surface but do not ensure complete observation.
- Valid JSON is not validated truth.
- Timestamps are estimates until a human verifies them.
- Use only appropriately licensed and classified video.

**Out of scope:** long-video segmentation, transcription, batch processing, embeddings, and production deployment.


## A realistic failure to investigate

A valid response may invent intent, identity, causality, or speech that is not visually observable. A model can also miss a short event while assigning precise timestamps to longer actions. Keyword or schema checks cannot catch those failures. Compare against a human-authored visible-evidence reference and require uncertainty for ambiguous details.


## Guided exercise

Modify one supplied fixture to trigger a realistic failure. Predict which validator or policy layer should catch it, run the reduced path, and report the observed category plus one remaining risk.

## Check your work

Try the exercise before expanding the reference solution.

In [ ]:
print('Expected method: change one fixture, preserve mode and policy, and identify the earliest rejecting boundary.')

## Final concept map

Approved clip → local validation → capability/model selection → bounded multimodal request → raw response → JSON parsing → schema checks → temporal matching → semantic support and privacy review → human decision. Each layer catches a different class of failure.


## Homework

Run one controlled live comparison or agent/video failure matrix. Submit configuration without secrets, structured outcomes, one success, one failure, and a limitation. The full rubric is in `homework/`.

## Glossary

| Term | Meaning in this lesson |
|---|---|
| **frame sampling** | Selecting a subset of video frames for model processing. |
| **multimodal request** | A request containing text plus media content. |
| **structured output** | A response constrained to a documented field schema. |
| **temporal IoU** | Intersection-over-union for two time intervals. |
| **ground truth** | A human-authored reference under a stated annotation policy. |
| **unsupported inference** | A claim not justified by visible or otherwise approved evidence. |
| **uncertainty** | An explicit statement of details the evidence cannot establish. |
| **human review** | A required judgment step not replaced by structural validation. |


## Sources and further study

- [Clemson RCD LLM Service](https://docs.rcd.clemson.edu/llm/)
- [OpenAI-compatible local model API](https://docs.rcd.clemson.edu/llm/usage/api/)
- [Clemson acceptable-use guidance](https://docs.rcd.clemson.edu/llm/acceptable_use/)

Course prose, code, and SVG diagrams are original. The video is NASA SVS item 30628 and is credited in the asset manifest. Sources verify service contracts and responsible-use terminology.
